# Symbolic MIDI Generation Draft Workbook

This draft documents the current Assignment 2 pipeline for two symbolic music generation tasks: unconditioned MIDI generation and prefix-conditioned MIDI continuation. It is a working report draft, not the final exported submission.

## 1. Introduction and Task Definitions

The project treats symbolic MIDI generation as next-token language modeling over MIDI-derived event tokens. A shared model can support both required tasks:

- **Task 1: symbolic unconditioned generation.** Sample a new token sequence from a beginning seed and decode it to MIDI.
- **Task 2: symbolic prefix-conditioned continuation.** Encode a real MIDI prefix, use it as the prompt, and sample a continuation.

The main neural model is a GPT-2-style causal Transformer initialized from scratch with a custom MIDI vocabulary. No pretrained GPT-2 weights, pretrained music checkpoints, or GPT-2 text tokenizer are used.

## 2. Dataset and Preprocessing

The current draft uses the final-scale Nottingham MIDI run as the main route and includes a bounded MAESTRO MIDI-only experiment as an optional comparison. Audio was not used. Files are split into train/validation partitions, tokenized, and converted into fixed-length next-token windows.

### Dataset Summary

| dataset | file_count | train_files | valid_files | token_min | token_max | token_mean | train_windows | valid_windows | vocab_size |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| nottingham_final | 500 | 450 | 50 | 49 | 3251 | 283.708 | 601 | 123 | 512 |
| maestro_final | 120 | 96 | 24 | 429 | 5867 | 3421.6583333333333 | 2374 | 769 | 512 |

## 3. Tokenization

The primary representation is MidiTok REMI. REMI represents symbolic music with discrete musical events such as bar, position, pitch, velocity, and duration. This keeps the model in a language-modeling setting while still preserving musical timing structure.

A simple custom tokenizer remains the fallback for smoke tests if MidiTok decoding becomes unstable, but the current real-data runs use REMI successfully.

## 4. Markov / N-Gram Baseline

The Markov baseline estimates next-token probabilities from local token histories. It gives a simple, reliable reference point for valid MIDI generation and validation perplexity.

### Model Metrics

| dataset | markov_valid_perplexity | transformer_train_loss_last | transformer_valid_loss | transformer_valid_perplexity | transformer_params | block_size | steps_completed |
| --- | --- | --- | --- | --- | --- | --- | --- |
| nottingham_final | 42.090160235864666 | 0.7876027822494507 | 2.9285693168640137 | 18.70085634877218 | 3356160 | 256 | 3000 |
| maestro_final | 281.79881948528003 | 4.671778678894043 | 4.744802231691321 | 114.98506285289204 | 478720 | 128 | 298 |

## 5. GPT2-Style Causal Transformer Trained From Scratch

The neural model uses `GPT2Config` and `GPT2LMHeadModel(config)` as a decoder-only Transformer architecture. The model is randomly initialized and trained on MIDI token windows. The Nottingham final-scale run used 500 MIDI files, a 450/50 train/validation split, `block_size=256`, `n_embd=256`, `n_layer=4`, `n_head=4`, dropout `0.1`, and 3,000 training steps. Its best checkpoint reached validation loss 2.9286 and validation perplexity 18.7009.

## 6. Optional MAESTRO MIDI-Only Experiment

A bounded MAESTRO MIDI-only comparison was run from local data. The subset used 120 short MIDI files selected from local metadata, with audio excluded. This run is useful context, but Nottingham remains the recommended final route because its final-scale Transformer has much stronger validation perplexity and longer selected candidates.

## 7. Reproducible Training Commands

The current training workflow supports full or bounded dataset runs, best-checkpoint saving, resume from checkpoint, checkpoint-only candidate generation, and candidate ranking. Long training should be run manually from PowerShell using the commands in `docs/training_commands.md`; this notebook is a draft report and does not launch long jobs itself.

## 8. Task 1: Symbolic Unconditioned Generation

For unconditioned generation, the sampler starts from a short seed and generates new MIDI tokens. Candidate files are decoded, parsed, and ranked by validity, note count, duration, pitch range, polyphony, and repetition heuristics.

## 9. Task 2: Prefix-Conditioned Continuation

For conditioned continuation, a validation MIDI prefix is used as the prompt. The model samples additional tokens after that prefix, and the resulting sequence is decoded to MIDI. This tests whether the same next-token model can generate in context.

## 10. Evaluation

Candidate MIDI files are checked for parseability and nonzero notes. The ranking table below is a rough quantitative screen, not a substitute for listening.

### Candidate Ranking

| path | dataset | task_type | model_type | temperature | top_k | candidate_index | valid | note_count | duration_seconds | notes_per_second | pitch_min | pitch_max | pitch_range | unique_pitch_count | max_simultaneous_notes | repeated_pitch_bigram_rate | score |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\markov_conditioned.mid | nottingham_final | conditioned | markov | nan | nan | nan | True | 257 | 88.25 | 2.912181303116147 | 56 | 83 | 27 | 22 | 3 | 0.6328125 | 1.3281033600881336 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\markov_unconditioned.mid | nottingham_final | unconditioned | markov | nan | nan | nan | True | 270 | 88.0 | 3.0681818181818183 | 60 | 83 | 23 | 21 | 5 | 0.6394052044609665 | 1.177628985017461 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk20_idx00.mid | nottingham_final | conditioned | transformer | 0.7 | 20.0 | 0.0 | True | 103 | 42.0 | 2.452380952380953 | 64 | 81 | 17 | 11 | 5 | 0.6764705882352942 | 0.1618405695611577 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk20_idx01.mid | nottingham_final | conditioned | transformer | 0.7 | 20.0 | 1.0 | True | 297 | 54.5 | 5.4495412844036695 | 64 | 79 | 15 | 11 | 181 | 0.8817567567567568 | -43.50872248395184 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk20_idx02.mid | nottingham_final | conditioned | transformer | 0.7 | 20.0 | 2.0 | True | 290 | 45.75 | 6.33879781420765 | 64 | 81 | 17 | 12 | 188 | 0.8823529411764706 | -45.43496999892854 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk50_idx00.mid | nottingham_final | conditioned | transformer | 0.7 | 50.0 | 0.0 | True | 127 | 69.25 | 1.8339350180505416 | 64 | 79 | 15 | 11 | 3 | 0.6746031746031746 | 0.6057450146123431 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk50_idx01.mid | nottingham_final | conditioned | transformer | 0.7 | 50.0 | 1.0 | True | 295 | 47.75 | 6.178010471204188 | 64 | 81 | 17 | 12 | 191 | 0.891156462585034 | -46.15623671807292 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p7_topk50_idx02.mid | nottingham_final | conditioned | transformer | 0.7 | 50.0 | 2.0 | True | 295 | 45.75 | 6.448087431693989 | 64 | 79 | 15 | 11 | 191 | 0.8877551020408163 | -46.267723318835735 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p8_topk20_idx00.mid | nottingham_final | conditioned | transformer | 0.8 | 20.0 | 0.0 | True | 115 | 42.0 | 2.738095238095238 | 64 | 81 | 17 | 12 | 8 | 0.631578947368421 | 0.2837667084377613 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p8_topk20_idx01.mid | nottingham_final | conditioned | transformer | 0.8 | 20.0 | 1.0 | True | 111 | 59.75 | 1.8577405857740583 | 64 | 81 | 17 | 11 | 2 | 0.6454545454545455 | 0.517613108068129 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p8_topk20_idx02.mid | nottingham_final | conditioned | transformer | 0.8 | 20.0 | 2.0 | True | 304 | 46.0 | 6.608695652173913 | 64 | 79 | 15 | 10 | 197 | 0.8877887788778878 | -47.78910412780408 |
| C:\Users\GD\OneDrive\Desktop\CSE253\assignment2-music-generation\outputs\candidates\nottingham_final\transformer_conditioned_temp0p8_topk50_idx00.mid | nottingham_final | conditioned | transformer | 0.8 | 50.0 | 0.0 | True | 98 | 51.75 | 1.893719806763285 | 64 | 79 | 15 | 10 | 1 | 0.6804123711340206 | 0.2273998954131182 |

### Selected Current Candidates

- nottingham_final unconditioned: `outputs\candidates\selected\nottingham_final\unconditioned_transformer.mid` (source `outputs\candidates\nottingham_final\transformer_unconditioned_temp1p0_topk20_idx00.mid`)
- nottingham_final conditioned: `outputs\candidates\selected\nottingham_final\conditioned_transformer.mid` (source `outputs\candidates\nottingham_final\transformer_conditioned_temp0p8_topk50_idx01.mid`)
- maestro_final unconditioned: `outputs\candidates\selected\maestro_final\unconditioned_transformer.mid` (source `outputs\candidates\maestro_final\transformer_unconditioned_temp1p0_topk50_idx00.mid`)
- maestro_final conditioned: `outputs\candidates\selected\maestro_final\conditioned_transformer.mid` (source `outputs\candidates\maestro_final\transformer_conditioned_temp0p8_topk20_idx01.mid`)

### Token Length Distributions

![Nottingham final token length distribution](../outputs/evaluation/figures/nottingham_final_token_lengths.png)

_Nottingham final token length distribution_

![MAESTRO final token length distribution](../outputs/evaluation/figures/maestro_final_token_lengths.png)

_MAESTRO final token length distribution_

### Pitch-Class Histograms

![Nottingham final train vs selected generated pitch-class histogram](../outputs/evaluation/figures/nottingham_final_pitch_class_histogram.png)

_Nottingham final train vs selected generated pitch-class histogram_

![MAESTRO final train vs selected generated pitch-class histogram](../outputs/evaluation/figures/maestro_final_pitch_class_histogram.png)

_MAESTRO final train vs selected generated pitch-class histogram_

## 11. Related Work Notes

This project is aligned with symbolic music generation methods from the course material, especially next-event prediction over symbolic music representations. The most relevant references for the final writeup are REMI / Pop Music Transformer, Music Transformer, Performance RNN-style symbolic sequence modeling, Markov and n-gram baselines, Nottingham, and MAESTRO.

## 12. Discussion, Limitations, and Future Work

The pipeline now produces valid MIDI candidates for both tasks. The main limitations are musical quality, heuristic candidate selection, and the fact that validation perplexity does not directly measure whether a melody is aesthetically satisfying. The next pass should listen to the selected files and add qualitative observations.

## 13. Current Artifacts and Remaining Submission Steps

Current generated artifacts live under `outputs/`, including metrics tables, figures, and selected candidate MIDI files. These are draft artifacts only. Final submission files have not been created yet.

Before submission, export this workbook to HTML, copy the selected MIDI files into `submission/` with the required names, and add the video URL file after recording the presentation.